Оценить эффективность пролонгации проектов по аккаунт-менеджерам и по отделу в целом.

Рассчитать:

— коэффициент пролонгации в первый месяц

— коэффициент пролонгации во второй месяц

Построить аналитический отчёт для принятия управленческих решений.

In [1]:
import pandas as pd
from scipy import stats as st
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize 
import warnings
warnings.filterwarnings("ignore")

#### Загрузка данных

In [2]:
df_prolongations = pd.read_csv(r"C:\Users\vrale\Downloads\prolongations.csv")
df_prolongations.head()

,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич
3,87,ноябрь 2022,Соколова Анастасия Викторовна
4,429,ноябрь 2022,Соколова Анастасия Викторовна


In [3]:
df_financial_data = pd.read_csv(r"C:\Users\vrale\Downloads\financial_data.csv")
df_financial_data.head()

,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
3,594,NaN,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
4,665,NaN,"10 000,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


#### Объединение данных

In [4]:
df = df_financial_data.merge(df_prolongations, how = 'right', on = 'id')
df.head()

,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,...,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account,month,AM
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович,ноябрь 2022,Васильев Артем Александрович
1,453,NaN,в ноль,"39 245,00","44 320,00","177 635,00",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович,ноябрь 2022,Васильев Артем Александрович
2,548,NaN,"674 000,00","674 000,00","674 000,00","674 000,00",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Михайлов Андрей Сергеевич,ноябрь 2022,Михайлов Андрей Сергеевич
3,87,основные работы,"70 050,00",NaN,"73 380,00","83 480,00","89 300,00","89 300,00","78 605,00","72 485,00",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Иванова Мария Сергеевна,ноябрь 2022,Соколова Анастасия Викторовна
4,429,NaN,"30 280,00","35 580,00","35 830,00","42 830,00",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Соколова Анастасия Викторовна,ноябрь 2022,Соколова Анастасия Викторовна


#### Преобразование в длинный формат

Изначально данные представлены в широком формате (месяцы — в колонках).

In [5]:
df_long = df.melt(
    id_vars=['id', 'month', 'AM'],
    value_vars=[
        'Ноябрь 2022', 'Декабрь 2022', 'Январь 2023',
        'Февраль 2023', 'Март 2023', 'Апрель 2023',
        'Май 2023', 'Июнь 2023', 'Июль 2023',
        'Август 2023', 'Сентябрь 2023', 'Октябрь 2023',
        'Ноябрь 2023', 'Декабрь 2023'
    ],
    var_name='revenue_month',
    value_name='revenue'
)

In [6]:
df_long.head()

,id,month,AM,revenue_month,revenue
0,42,ноябрь 2022,Васильев Артем Александрович,Ноябрь 2022,"36 220,00"
1,453,ноябрь 2022,Васильев Артем Александрович,Ноябрь 2022,в ноль
2,548,ноябрь 2022,Михайлов Андрей Сергеевич,Ноябрь 2022,"674 000,00"
3,87,ноябрь 2022,Соколова Анастасия Викторовна,Ноябрь 2022,"70 050,00"
4,429,ноябрь 2022,Соколова Анастасия Викторовна,Ноябрь 2022,"30 280,00"


#### Очистка данных
В данных присутствуют:
— строки вида "36 220,00"
— текстовые значения ("в ноль", "стоп")

Приводим всё к числовому формату.

In [7]:
df_long['revenue'] = (
    df_long['revenue']
    .astype(str)
    .str.replace('\xa0', '')
    .str.replace(' ', '')
    .str.replace(',', '.')
    .str.strip()
)

In [8]:
df_long['revenue'] = df_long['revenue'].replace({
    'вноль': '0',
    'в ноль': '0',
    'nan': None,
    'None': None,
    '': None
})

In [9]:
df_long['revenue'] = pd.to_numeric(df_long['revenue'], errors='coerce')

In [10]:
df_long.head()

,id,month,AM,revenue_month,revenue
0,42,ноябрь 2022,Васильев Артем Александрович,Ноябрь 2022,36220.0
1,453,ноябрь 2022,Васильев Артем Александрович,Ноябрь 2022,0.0
2,548,ноябрь 2022,Михайлов Андрей Сергеевич,Ноябрь 2022,674000.0
3,87,ноябрь 2022,Соколова Анастасия Викторовна,Ноябрь 2022,70050.0
4,429,ноябрь 2022,Соколова Анастасия Викторовна,Ноябрь 2022,30280.0


#### Преобразование дат

Месяцы представлены на русском языке, поэтому переводим в datetime.

In [11]:
months_dict = {
    'Январь': 'January',
    'Февраль': 'February',
    'Март': 'March',
    'Апрель': 'April',
    'Май': 'May',
    'Июнь': 'June',
    'Июль': 'July',
    'Август': 'August',
    'Сентябрь': 'September',
    'Октябрь': 'October',
    'Ноябрь': 'November',
    'Декабрь': 'December'
}

In [12]:
df_long['revenue_month'] = (
    df_long['revenue_month']
    .replace(months_dict, regex=True)
)

df_long['revenue_month'] = pd.to_datetime(
    df_long['revenue_month'],
    format='%B %Y',
    errors='coerce'
)

In [13]:
months_dict = {
    'январь': 'January',
    'февраль': 'February',
    'март': 'March',
    'апрель': 'April',
    'май': 'May',
    'июнь': 'June',
    'июль': 'July',
    'август': 'August',
    'сентябрь': 'September',
    'октябрь': 'October',
    'ноябрь': 'November',
    'декабрь': 'December'
}

In [14]:
df_long['month'] = (
    df_long['month']
    .replace(months_dict, regex=True)
)

df_long['end_month'] = pd.to_datetime(
    df_long['month'],
    format='%B %Y',
    errors='coerce'
)

#### Добавление будущих месяцев
Для определения пролонгации считаем:

— месяц + 1

— месяц + 2

In [15]:
df_long['next_month'] = df_long['end_month'] + pd.DateOffset(months=1)
df_long['next_2_month'] = df_long['end_month'] + pd.DateOffset(months=2)

In [16]:
df_long.head()

,id,month,AM,revenue_month,revenue,end_month,next_month,next_2_month
0,42,November 2022,Васильев Артем Александрович,2022-11-01,36220.0,2022-11-01,2022-12-01,2023-01-01
1,453,November 2022,Васильев Артем Александрович,2022-11-01,0.0,2022-11-01,2022-12-01,2023-01-01
2,548,November 2022,Михайлов Андрей Сергеевич,2022-11-01,674000.0,2022-11-01,2022-12-01,2023-01-01
3,87,November 2022,Соколова Анастасия Викторовна,2022-11-01,70050.0,2022-11-01,2022-12-01,2023-01-01
4,429,November 2022,Соколова Анастасия Викторовна,2022-11-01,30280.0,2022-11-01,2022-12-01,2023-01-01


#### Self join (поиск будущей выручки)

Соединяем таблицу саму с собой, чтобы получить выручку будущих месяцев.

In [17]:
df_next = df_long[['id', 'revenue_month', 'revenue']].rename(columns={
    'revenue_month': 'next_month',
    'revenue': 'next_revenue'
})

In [18]:
df = df_long.merge(
    df_next,
    on=['id', 'next_month'],
    how='left'
)

In [19]:
df_next2 = df_long[['id', 'revenue_month', 'revenue']].rename(columns={
    'revenue_month': 'next_2_month',
    'revenue': 'next_2_revenue'
})

In [20]:
df = df.merge(
    df_next2,
    on=['id', 'next_2_month'],
    how='left'
)

In [21]:
df

,id,month,AM,revenue_month,revenue,end_month,next_month,next_2_month,next_revenue,next_2_revenue
0,42,November 2022,Васильев Артем Александрович,2022-11-01,36220.0,2022-11-01,2022-12-01,2023-01-01,NaN,NaN
1,453,November 2022,Васильев Артем Александрович,2022-11-01,0.0,2022-11-01,2022-12-01,2023-01-01,39245.0,44320.0
2,453,November 2022,Васильев Артем Александрович,2022-11-01,0.0,2022-11-01,2022-12-01,2023-01-01,39245.0,44320.0
3,453,November 2022,Васильев Артем Александрович,2022-11-01,0.0,2022-11-01,2022-12-01,2023-01-01,39245.0,44320.0
4,453,November 2022,Васильев Артем Александрович,2022-11-01,0.0,2022-11-01,2022-12-01,2023-01-01,39245.0,44320.0
...,...,...,...,...,...,...,...,...,...,...
307743,1004,December 2023,без А/М,2023-12-01,0.0,2023-12-01,2024-01-01,2024-02-01,NaN,NaN
307744,281,December 2023,Соколова Анастасия Викторовна,2023-12-01,74350.0,2023-12-01,2024-01-01,2024-02-01,NaN,NaN
307745,281,December 2023,Соколова Анастасия Викторовна,2023-12-01,NaN,2023-12-01,2024-01-01,2024-02-01,NaN,NaN
307746,785,December 2023,Соколова Анастасия Викторовна,2023-12-01,0.0,2023-12-01,2024-01-01,2024-02-01,NaN,NaN


Оставляем только строки, где проект завершился.
Это важно, чтобы избежать дублирования.

In [22]:
df_base = df[df['revenue_month'] == df['end_month']]

#### Пролонгация
1 месяц → есть выручка сразу
2 месяц → не было в первый, но появилась во второй

In [23]:
df_base['prolong_1'] = df_base['next_revenue'] > 0

In [24]:
df_base['prolong_2'] = (df_base['next_revenue'].isna()) & (df_base['next_2_revenue'] > 0)

#### Денежные метрики

Считаем через выручку 

In [25]:
df_base['revenue_prolong_1m'] = df_base['revenue'].where(df_base['prolong_1'], 0) 

df_base['revenue_not_prolong_1m'] = df_base['revenue'].where(~df_base['prolong_1'], 0)

df_base['revenue_prolong_2m'] = df_base['revenue'].where(df_base['prolong_2'], 0)

#### Агрегация по менеджерам

In [26]:
agg = df_base.groupby(['AM', 'end_month'], as_index=False).agg({
    'revenue': 'sum',
    'revenue_prolong_1m': 'sum',
    'revenue_not_prolong_1m': 'sum',
    'revenue_prolong_2m': 'sum'
})

#### Расчёт коэффициентов

In [27]:
agg['coef_prolong_1m'] = (agg['revenue_prolong_1m'] / agg['revenue'])

In [28]:
agg['coef_prolong_2m'] = (agg['revenue_prolong_2m'] / agg['revenue_not_prolong_1m'])
agg

,AM,end_month,revenue,revenue_prolong_1m,revenue_not_prolong_1m,revenue_prolong_2m,coef_prolong_1m,coef_prolong_2m
0,Васильев Артем Александрович,2022-11-01,30845288.00,29929700.00,915588.00,0.00,0.970317,0.000000
1,Васильев Артем Александрович,2022-12-01,78910854.88,26594165.44,52316689.44,26585794.72,0.337015,0.508170
2,Васильев Артем Александрович,2023-01-01,66848700.00,66047500.00,801200.00,557600.00,0.988015,0.695956
3,Васильев Артем Александрович,2023-02-01,42079262.00,37978897.00,4100365.00,2345040.00,0.902556,0.571910
4,Васильев Артем Александрович,2023-03-01,49157226.80,9168597.50,39988629.30,0.00,0.186516,0.000000
...,...,...,...,...,...,...,...,...
81,Соколова Анастасия Викторовна,2023-11-01,5772580.00,4280510.00,1492070.00,0.00,0.741525,0.000000
82,Соколова Анастасия Викторовна,2023-12-01,878866.21,0.00,878866.21,0.00,0.000000,0.000000
83,Федорова Марина Васильевна,2023-12-01,50250.00,0.00,50250.00,0.00,0.000000,0.000000
84,без А/М,2023-07-01,0.00,0.00,0.00,0.00,NaN,NaN


#### Агрегация по отделу

In [29]:
dept = df_base.groupby(['end_month'], as_index=False).agg({
    'revenue': 'sum',
    'revenue_prolong_1m': 'sum',
    'revenue_not_prolong_1m': 'sum',
    'revenue_prolong_2m': 'sum'
})

dept['coef_prolong_1m'] = dept['revenue_prolong_1m'] / dept['revenue']
dept['coef_prolong_2m'] = dept['revenue_prolong_2m'] / dept['revenue_not_prolong_1m']

In [30]:
dept

,end_month,revenue,revenue_prolong_1m,revenue_not_prolong_1m,revenue_prolong_2m,coef_prolong_1m,coef_prolong_2m
0,2022-11-01,4.273430e+07,3.908952e+07,3644778.00,1533180.00,0.914711,0.420651
1,2022-12-01,1.868476e+08,8.793725e+07,98910377.20,46207284.72,0.470636,0.467163
2,2023-01-01,9.847028e+07,7.749067e+07,20979611.50,2035850.00,0.786945,0.097039
3,2023-02-01,8.221267e+07,6.294541e+07,19267262.50,8574283.75,0.765641,0.445018
4,2023-03-01,1.188376e+08,5.766804e+07,61169576.80,10620463.75,0.485268,0.173623
5,2023-04-01,1.938969e+08,1.634004e+08,30496484.00,13076440.00,0.842718,0.428785
6,2023-05-01,1.082142e+07,5.124656e+06,5696768.07,488771.35,0.473566,0.085798
7,2023-06-01,1.127964e+08,5.480090e+07,57995506.10,7715945.00,0.485839,0.133044
8,2023-07-01,6.287977e+07,4.398876e+07,18891008.03,1282823.75,0.699569,0.067907
9,2023-08-01,6.453288e+07,1.810248e+07,46430406.37,3624020.00,0.280516,0.078053


#### Агрегация по отделам и годам

In [31]:
df_base['year'] = df_base['end_month'].dt.year

In [32]:
year_manager = df_base.groupby(['AM', 'year'], as_index=False).agg({
    'revenue': 'sum',
    'revenue_prolong_1m': 'sum',
    'revenue_not_prolong_1m': 'sum',
    'revenue_prolong_2m': 'sum'
})

In [33]:
year_manager['coef_prolong_1m'] = year_manager['revenue_prolong_1m'] / year_manager['revenue']

year_manager['coef_prolong_2m'] = year_manager['revenue_prolong_2m'] / year_manager['revenue_not_prolong_1m']

In [34]:
year_dept = df_base.groupby(['year'], as_index=False).agg({
    'revenue': 'sum',
    'revenue_prolong_1m': 'sum',
    'revenue_not_prolong_1m': 'sum',
    'revenue_prolong_2m': 'sum'
})

In [35]:
year_dept['coef_prolong_1m'] = year_dept['revenue_prolong_1m'] / year_dept['revenue']

year_dept['coef_prolong_2m'] = year_dept['revenue_prolong_2m'] / year_dept['revenue_not_prolong_1m']

p.s. Я агрегировала данные по годам, суммируя выручку и рассчитывая коэффициенты пролонгации на основе агрегированных значений, что обеспечивает корректность метрик.

In [36]:
with pd.ExcelWriter('prolongation_report.xlsx') as writer:
    agg.to_excel(writer, sheet_name='Managers', index=False)
    dept.to_excel(writer, sheet_name='Data', index=False)
    year_manager.to_excel(writer, sheet_name='year_manager', index=False)
    year_dept.to_excel(writer, sheet_name='year_dept', index=False)